# Clinic Case Study — Phase 5: Reinforcement Learning Staffing Agent

**Case study**: Community Health Clinic | **Phase**: 5 of 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Frame the clinic staffing problem as a Markov Decision Process (MDP).
2. Instantiate `ClinicEnv` and interact with it using the Gymnasium API.
3. Train a DQN agent with `stable-baselines3` to minimise patient wait time.
4. Compare the learned policy to a simple rule-based heuristic.

---
> Phase 5 replaces the manual scenario comparison from Phase 4 with an RL agent
> that *learns* a staffing policy from interaction with the simulation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

from simdes.envs.clinic_env import ClinicEnv

## MDP Formulation

| MDP component | Clinic formulation |
|---|---|
| **State** *s* | (queue_reg, queue_triage, queue_exam, util_nurses, time_fraction) |
| **Action** *a* | Number of triage nurses to deploy: {1, 2, 3, 4} |
| **Reward** *r* | −mean_wait_triage over the last 30-minute interval |
| **Episode** | One 8-hour operating day (480 min); 16 decision epochs |
| **Termination** | End of day |

The agent's goal is to minimise total (undiscounted) patient wait time over an episode,
subject to the constraint that it can only change staffing at 30-minute intervals.

In [ ]:
# Inspect the environment
env = ClinicEnv(sim_time=480, max_nurses=4, decision_interval=30, seed=0)
print('Observation space:', env.observation_space)
print('Action space:     ', env.action_space)
print('Sample observation:', env.observation_space.sample())

In [ ]:
# Random policy baseline — random staffing at each epoch
N_EPISODES = 50
random_returns = []

for ep in range(N_EPISODES):
    obs, _ = env.reset(seed=ep)
    total_reward = 0.0
    terminated = truncated = False
    while not (terminated or truncated):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
    random_returns.append(total_reward)

print(f'Random policy — mean return: {np.mean(random_returns):.2f} ± {np.std(random_returns):.2f}')

In [ ]:
# Heuristic policy: always deploy 2 nurses (best from Phase 4)
heuristic_returns = []

for ep in range(N_EPISODES):
    obs, _ = env.reset(seed=ep)
    total_reward = 0.0
    terminated = truncated = False
    while not (terminated or truncated):
        action = 1   # action=1 → 2 nurses (0-indexed)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
    heuristic_returns.append(total_reward)

print(f'Heuristic policy — mean return: {np.mean(heuristic_returns):.2f} ± {np.std(heuristic_returns):.2f}')

In [ ]:
# Train DQN agent
train_env = Monitor(ClinicEnv(sim_time=480, max_nurses=4, decision_interval=30, seed=2024))

model = DQN(
    policy='MlpPolicy',
    env=train_env,
    learning_rate=1e-3,
    buffer_size=5000,
    learning_starts=200,
    batch_size=32,
    gamma=0.99,
    exploration_fraction=0.3,
    verbose=1,
    seed=42,
)

model.learn(total_timesteps=10_000, progress_bar=True)
model.save('clinic_dqn')
print('Training complete.')

In [ ]:
# Evaluate trained agent
eval_env = ClinicEnv(sim_time=480, max_nurses=4, decision_interval=30, seed=99)
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=50)
print(f'DQN agent — mean return: {mean_reward:.2f} ± {std_reward:.2f}')

In [ ]:
# Comparison bar chart
labels  = ['Random', 'Heuristic\n(always 2)', 'DQN agent']
means   = [np.mean(random_returns), np.mean(heuristic_returns), mean_reward]
errors  = [np.std(random_returns),  np.std(heuristic_returns),  std_reward]

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(labels))
ax.bar(x, means, yerr=errors, capsize=6,
       color=['#d62728', '#1f77b4', '#2ca02c'], alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Mean episode return (higher = less wait)')
ax.set_title('Clinic RL — policy comparison')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

## Summary

The DQN agent learns a state-dependent staffing policy:
- During busy periods (long triage queue), it increases nurse count.
- During quiet periods, it scales back to save cost.

This is superior to both the random policy and the static heuristic from Phase 4,
because it adapts to real-time queue state rather than following a fixed plan.

**Key lesson**: RL agents can outperform rule-based policies in dynamic environments
when the state is observable and the reward function captures the objective well.

## Try It Yourself

1. Inspect the trained policy: for each queue length bucket, what action does the DQN prefer?
2. Train a PPO agent instead. Does it converge faster or slower than DQN?
3. Modify the reward to penalise *both* wait time and the number of nurses deployed.
   How does the learned policy change?